# Experiment-4: Term Frequency and NER (Local Version)

### Setup (local only)
Run this once in a terminal before starting:
```
pip install nltk spacy pandas
python -m spacy download en_core_web_sm
```

**Input data for 4.1 and 4.2:** the assignment links a Google Drive file. Colab could read it directly because Drive was mounted there — locally you need to download that file yourself first:
1. Open the link, download the file to your computer (e.g. into the same folder as this notebook).
2. Set `INPUT_FILE_PATH` below to wherever you saved it.

If the file turns out to be a `.txt`, `open()` below works as-is. If it's actually a `.docx` or `.pdf`, tell me and I'll adjust the reading code.

In [5]:
# Update this to wherever you saved the downloaded input file
INPUT_FILE_PATH = r"C:\Rudransh\CLNLP_Lab\Experiment4\input.txt"

### Experiment 4.1: Term-Frequency Analysis and Named Entity Recognition (using a toolkit)
- Calculate the frequency of each word and save the output in CSV format (Term, Frequency)
- Identify the top 10 most frequent terms and display their corresponding frequencies.
- Perform Named Entity Recognition using spaCy.

In [6]:
import spacy
import csv
from collections import Counter

nlp = spacy.load('en_core_web_sm')

with open(INPUT_FILE_PATH, 'r', encoding='utf-8') as f:
    text_4_1 = f.read()

doc = nlp(text_4_1)

# --- Term Frequency (using spaCy tokenization) ---
terms = [t.text.lower() for t in doc if t.is_alpha]
freq = Counter(terms)

csv_path_4_1 = "4.1_term_frequency.csv"
with open(csv_path_4_1, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Term', 'Frequency'])
    for term, count in freq.most_common():
        writer.writerow([term, count])

print(f"Saved term frequencies to {csv_path_4_1}")

print("\nTop 10 Most Frequent Terms:")
for term, count in freq.most_common(10):
    print(f"{term:<15} -> {count}")

# --- Named Entity Recognition ---
print("\nNamed Entities Found:")
if doc.ents:
    for ent in doc.ents:
        print(f"{ent.text:<25} -> {ent.label_}")
else:
    print("(none found)")

Saved term frequencies to 4.1_term_frequency.csv

Top 10 Most Frequent Terms:
a               -> 88
and             -> 86
the             -> 75
of              -> 42
to              -> 42
in              -> 42
can             -> 42
is              -> 37
may             -> 30
word            -> 30

Named Entities Found:
Natural Language Processing and Artificial Intelligence

Natural Language Processing -> ORG
NLP                       -> ORG
English                   -> LANGUAGE
Hindi                     -> GPE
Assamese                  -> NORP
Bengali                   -> NORP
Tamil                     -> GPE
NLP                       -> ORG
NLP                       -> ORG
NLP                       -> ORG
twenty                    -> CARDINAL
twenty                    -> DATE
Python                    -> GPE
First                     -> ORDINAL
first                     -> ORDINAL
thousands                 -> CARDINAL
hundreds or thousands     -> CARDINAL
twenty                    ->

### Experiment 4.2: Term-Frequency Analysis (without using any toolkit)
- Calculate the frequency of each word and save the output in CSV format (Term, Frequency)
- Identify the top 10 most frequent terms and display their corresponding frequencies.

In [7]:
import string
import csv
from collections import Counter

with open(INPUT_FILE_PATH, 'r', encoding='utf-8') as f:
    text_4_2 = f.read()

text_4_2 = text_4_2.lower()
text_4_2 = "".join(c for c in text_4_2 if c not in string.punctuation)
tokens_4_2 = text_4_2.split()

freq_4_2 = Counter(tokens_4_2)

csv_path_4_2 = "4.2_term_frequency_no_toolkit.csv"
with open(csv_path_4_2, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Term', 'Frequency'])
    for term, count in freq_4_2.most_common():
        writer.writerow([term, count])

print(f"Saved term frequencies to {csv_path_4_2}")

print("\nTop 10 Most Frequent Terms:")
for term, count in freq_4_2.most_common(10):
    print(f"{term:<15} -> {count}")

Saved term frequencies to 4.2_term_frequency_no_toolkit.csv

Top 10 Most Frequent Terms:
a               -> 90
and             -> 86
the             -> 75
to              -> 42
in              -> 42
can             -> 41
of              -> 40
is              -> 37
may             -> 30
word            -> 30


### Experiment 4.3: TF-IDF from scratch (no NLTK, spaCy, or any NLP toolkit)
- Read multiple text documents, lowercase, remove punctuation, tokenize
- Calculate TF, DF, IDF (log formula), and TF-IDF for every term in every document
- Display Term, TF, DF, IDF, and TF-IDF at each step
- Identify the top 10 terms with the highest TF-IDF score for each document

Uses the three documents given in the assignment directly (no external file needed).

In [8]:
import string
import math
from collections import Counter

documents = [
    "Natural language processing is a field of artificial intelligence.",
    "Natural language processing helps computers understand human language.",
    "Machine learning is an important part of artificial intelligence."
]

def preprocess(text):
    text = text.lower()
    text = "".join(c for c in text if c not in string.punctuation)
    return text.split()

tokenized_docs = [preprocess(doc) for doc in documents]

# --- Term Frequency (TF) per document ---
tf_per_doc = []
for tokens in tokenized_docs:
    counts = Counter(tokens)
    total = len(tokens)
    tf_per_doc.append({term: count / total for term, count in counts.items()})

# --- Document Frequency (DF) across all documents ---
all_terms = sorted(set(term for tokens in tokenized_docs for term in tokens))
df = {term: sum(1 for tokens in tokenized_docs if term in tokens) for term in all_terms}

# --- Inverse Document Frequency (IDF), log formula ---
N = len(documents)
idf = {term: math.log(N / df[term]) for term in all_terms}

# --- TF-IDF per document ---
tfidf_per_doc = []
for tf in tf_per_doc:
    tfidf_per_doc.append({term: tf_val * idf[term] for term, tf_val in tf.items()})

# --- Display Term, TF, DF, IDF, TF-IDF for each document ---
for i, tokens in enumerate(tokenized_docs, start=1):
    print(f"\n=== Document {i} ===")
    print(f"{'Term':<15}{'TF':<10}{'DF':<6}{'IDF':<10}{'TF-IDF':<10}")
    for term in sorted(set(tokens)):
        print(f"{term:<15}{tf_per_doc[i-1][term]:<10.4f}{df[term]:<6}{idf[term]:<10.4f}{tfidf_per_doc[i-1][term]:<10.4f}")

# --- Top 10 terms by TF-IDF for each document ---
for i, scores in enumerate(tfidf_per_doc, start=1):
    top10 = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:10]
    print(f"\nTop {min(10, len(top10))} TF-IDF terms in Document {i}:")
    for term, score in top10:
        print(f"{term:<15} -> {score:.4f}")


=== Document 1 ===
Term           TF        DF    IDF       TF-IDF    
a              0.1111    1     1.0986    0.1221    
artificial     0.1111    2     0.4055    0.0451    
field          0.1111    1     1.0986    0.1221    
intelligence   0.1111    2     0.4055    0.0451    
is             0.1111    2     0.4055    0.0451    
language       0.1111    2     0.4055    0.0451    
natural        0.1111    2     0.4055    0.0451    
of             0.1111    2     0.4055    0.0451    
processing     0.1111    2     0.4055    0.0451    

=== Document 2 ===
Term           TF        DF    IDF       TF-IDF    
computers      0.1250    1     1.0986    0.1373    
helps          0.1250    1     1.0986    0.1373    
human          0.1250    1     1.0986    0.1373    
language       0.2500    2     0.4055    0.1014    
natural        0.1250    2     0.4055    0.0507    
processing     0.1250    2     0.4055    0.0507    
understand     0.1250    1     1.0986    0.1373    

=== Document 3 ===
Term